# Bounded P4b task-segmented-objective smoke

Run this notebook with **Internet enabled**, a **Tesla T4 GPU**, and the private Kaggle secret `GITHUB_TOKEN`. Attach exactly these immutable inputs:

1. `thestonedape/task-aware-eegtotext`, **Version 2** (prompt-neutral vectors);
2. `thestonedape/task-aware-eeg2text-task-segmented-protocol`, **Version 1**;
3. `thestonedape/task-aware-eeg2text-task-segmented-schedule`, **Version 1**.

This executes only fold 0, seed 20260717, epoch 1, batches 0 and 1 for all three arms. It cannot evaluate checkpoints or confirmation data, cannot make a scientific decision, and cannot start the 45 fits. Do not delete `OUTPUT` on a retry: the runner validates and resumes or reuses it.


In [ ]:
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
IMPLEMENTATION_COMMIT = 'a9b214492a40c6ddb561ecd6cb279ff6cedd2a18'
WORKTREE = '/kaggle/working/SemKey'
OUTPUT = '/kaggle/working/task-aware-eeg2text-task-segmented-smoke'
VECTOR_DATASET = ('thestonedape/task-aware-eegtotext', 2)
PROTOCOL_DATASET = ('thestonedape/task-aware-eeg2text-task-segmented-protocol', 1)
SCHEDULE_DATASET = ('thestonedape/task-aware-eeg2text-task-segmented-schedule', 1)
assert len(IMPLEMENTATION_COMMIT) == 40


In [ ]:
import glob, hashlib, json, os, platform, shutil, subprocess, sys
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import numpy as np
import torch
from kaggle_secrets import UserSecretsClient

def digest(path):
    state = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            state.update(block)
    return state.hexdigest()

assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'
print({
    'python': platform.python_version(), 'numpy': np.__version__,
    'torch': torch.__version__, 'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0),
})
github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert github_token, 'Enable the private Kaggle Secret named GITHUB_TOKEN'
askpass = '/kaggle/working/git_askpass.py'
with open(askpass, 'w', encoding='utf-8') as handle:
    handle.write("#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in prompt else 'x-access-token')\n")
os.chmod(askpass, 0o700)
clone_env = os.environ.copy()
clone_env.update({'GIT_ASKPASS': askpass, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
if os.path.exists(WORKTREE):
    shutil.rmtree(WORKTREE)
try:
    subprocess.run(['git', 'clone', REPO_URL, WORKTREE], check=True, env=clone_env)
finally:
    if os.path.exists(askpass):
        os.remove(askpass)
subprocess.run(['git', '-C', WORKTREE, 'checkout', '--detach', IMPLEMENTATION_COMMIT], check=True)
actual_commit = subprocess.check_output(['git', '-C', WORKTREE, 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == IMPLEMENTATION_COMMIT
tests = [
    'project_adapters.test_task_segmented_objective',
    'evaluation.test_task_segmented_objective_protocol',
    'evaluation.test_task_segmented_training_schedule',
    'evaluation.test_verify_task_segmented_protocol_artifact',
    'evaluation.test_verify_task_segmented_training_schedule_output',
    'evaluation.test_verify_task_segmented_training_schedule_artifact',
    'evaluation.test_task_segmented_objective_runner',
    'evaluation.test_verify_task_segmented_smoke_artifact',
    'evaluation.test_decide_task_segmented_objective',
]
subprocess.run([sys.executable, '-m', 'unittest', *tests], check=True, cwd=WORKTREE)
print({'implementation_commit': actual_commit, 'regressions': 'PASS'})


In [ ]:
VECTOR_REQUIRED = {
    'pilot_input_manifest.json', 'run_metadata.json', 'eeg', 'text',
}
PROTOCOL_REQUIRED = {
    'batch_grid_feasibility.csv', 'candidate_pools.csv',
    'confirmation_donors.csv', 'outer_split_assignments.csv',
    'protocol_registry.json', 'pseudo_groups.csv', 'text_group_folds.csv',
    'task_segmented_protocol_report.json', 'protocol_freeze_run_metadata.json',
    'task_segmented_objective_contract.json',
}
SCHEDULE_REQUIRED = {
    'trial_catalog.csv', 'schedule_indices.u32le', 'schedule_units.csv',
    'schedule_audit.csv', 'task_segmented_training_schedule_manifest.json',
    'task_segmented_training_schedule_report.json',
    'task_segmented_training_schedule_contract.json',
    'parent_protocol_verification_report.json', 'schedule_freeze_run_metadata.json',
}

def exact_roots(marker, required):
    roots = []
    for marker_path in glob.glob('/kaggle/input/**/' + marker, recursive=True):
        root = os.path.dirname(marker_path)
        try:
            names = set(os.listdir(root))
        except OSError:
            continue
        if names == required and all(not os.path.islink(os.path.join(root, name)) for name in names):
            roots.append(root)
    return sorted(set(roots))

vector_roots = exact_roots('pilot_input_manifest.json', VECTOR_REQUIRED)
protocol_roots = exact_roots('task_segmented_protocol_report.json', PROTOCOL_REQUIRED)
schedule_roots = exact_roots('schedule_freeze_run_metadata.json', SCHEDULE_REQUIRED)
assert len(vector_roots) == 1, ('Attach exact prompt-neutral dataset Version 2', vector_roots)
assert len(protocol_roots) == 1, ('Attach exact protocol dataset Version 1', protocol_roots)
assert len(schedule_roots) == 1, ('Attach exact schedule dataset Version 1', schedule_roots)
VECTOR_ROOT, PROTOCOL_ROOT, SCHEDULE_ROOT = vector_roots[0], protocol_roots[0], schedule_roots[0]
print({
    'vector_dataset': VECTOR_DATASET, 'vector_root': VECTOR_ROOT,
    'protocol_dataset': PROTOCOL_DATASET, 'protocol_root': PROTOCOL_ROOT,
    'schedule_dataset': SCHEDULE_DATASET, 'schedule_root': SCHEDULE_ROOT,
})


In [ ]:
# OUTPUT is intentionally not deleted: a valid partial run resumes and a valid
# completed run is independently rehashed and reused. Binding drift hard-fails.
subprocess.run([
    sys.executable,
    os.path.join(WORKTREE, 'evaluation', 'run_task_segmented_objective.py'),
    '--vector-root', VECTOR_ROOT,
    '--protocol-root', PROTOCOL_ROOT,
    '--schedule-root', SCHEDULE_ROOT,
    '--output-root', OUTPUT,
    '--project-commit', IMPLEMENTATION_COMMIT,
    '--device', 'cuda',
    '--mode', 'bounded_smoke',
], check=True, cwd=WORKTREE)


In [ ]:
manifest_path = os.path.join(OUTPUT, 'task_segmented_smoke_manifest.json')
manifest_sha256 = digest(manifest_path)
report_path = '/kaggle/working/task-segmented-smoke-independent-verification.json'
subprocess.run([
    sys.executable,
    os.path.join(WORKTREE, 'evaluation', 'verify_task_segmented_smoke_artifact.py'),
    '--artifact-root', OUTPUT,
    '--expected-manifest-sha256', manifest_sha256,
    '--output-report', report_path,
], check=True, cwd=WORKTREE)
with open(report_path, encoding='utf-8') as handle:
    report = json.load(handle)
with open(manifest_path, encoding='utf-8') as handle:
    manifest = json.load(handle)
assert report['status'] == 'pass'
assert report['arms'] == ['global_mixed', 'true_task_segmented', 'pseudo_task_segmented']
assert report['optimizer_steps_per_arm'] == 2 and report['total_optimizer_steps'] == 6
assert report['common_trace_rows'] == 128 and report['checkpoint_deserialized'] is False
assert report['full_training_authorized'] is False
assert report['scientific_decision_permitted'] is False
assert report['held_out_test_accessed'] is False
assert manifest['project_commit'] == IMPLEMENTATION_COMMIT
assert manifest['authorization']['full_training_authorized'] is False
assert manifest['authorization']['checkpoint_or_confirmation_evaluation_authorized'] is False
assert manifest['authorization']['scientific_decision_permitted'] is False
assert manifest['authorization']['held_out_test_accessed'] is False
print({
    'status': report['status'], 'implementation_commit': IMPLEMENTATION_COMMIT,
    'smoke_manifest_sha256': manifest_sha256,
    'verification_report_sha256': digest(report_path),
    'arms': report['arms'], 'total_optimizer_steps': report['total_optimizer_steps'],
    'full_training_authorized': report['full_training_authorized'],
    'scientific_decision_permitted': report['scientific_decision_permitted'],
    'held_out_test_accessed': report['held_out_test_accessed'],
})
print('P4B TASK-SEGMENTED OBJECTIVE BOUNDED REAL-DATA SMOKE: PASS')


After the final PASS, use **Save Version**. Then preserve `/kaggle/working/task-aware-eeg2text-task-segmented-smoke` as a new **private** Kaggle dataset, preferably `task-aware-eeg2text-task-segmented-smoke`, and record the exact dataset slug and version. The 45 fits remain unauthorized until that preserved artifact passes a clean-remount verifier with an externally pinned manifest hash.
